In [4]:
import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from itertools import product, combinations


import sys
import importlib


sys.path.append(r"C:\Users\mattm\OneDrive\Desktop\Research\Projects\Triangle Lattice\Jupyter Notebooks\8Q_Triangle_Lattice_v1")
sys.path.append(r"C:\Users\mattm\OneDrive\Desktop\Research\Projects\Triangle Lattice\Jupyter Notebooks\8Q_Triangle_Lattice_v1\current_measurements")
sys.path.append(r"C:\Users\mattm\OneDrive\Desktop\Research\Projects\Triangle Lattice\Jupyter Notebooks\8Q_Triangle_Lattice_v1\current_simulations")


import current_measurements.src.src_current_measurement_simulations
importlib.reload(current_measurements.src.src_current_measurement_simulations);
from current_measurements.src.src_current_measurement_simulations import CurrentMeasurementSimulation

import current_measurements.src.src_current_measurement_simulations_particle_sector
importlib.reload(current_measurements.src.src_current_measurement_simulations_particle_sector);
from current_measurements.src.src_current_measurement_simulations_particle_sector import CurrentMeasurementSimulationParticleSector


from current_simulations.current_simulation import FockBasisState, convert_reduced_to_fock_state


# 1. Two Levels, no interactions

In [5]:
def generate_Hamiltonian(annihilation_operators, num_levels, num_qubits, J, J_parallel, U, frequencies):
    H = 0

    for i in range(num_qubits):
        a_i = annihilation_operators[i]
        H += frequencies[i] * a_i.dag() * a_i
        H += U / 2 * a_i.dag() * a_i * (a_i.dag() * a_i - 1)

        if i < num_qubits - 1:
            a_j = annihilation_operators[i + 1]
            H += J * (a_i.dag() * a_j + a_j.dag() * a_i)
        
        if i < num_qubits - 2:
            a_j = annihilation_operators[i + 2]
            H += J_parallel * (a_i.dag() * a_i) * (a_j.dag() * a_j)

    return H

In [6]:
num_levels = 2
num_qubits = 8
num_particles = 4

J = 6 * 2 * np.pi  # in MHz

U = 180* 2 * np.pi  # in MHz

resonance_point = 4e3 * 2 * np.pi  # in MHz
detunings = np.linspace(-200, 200, num_qubits) * 2 * np.pi  # in MHz

frequencies = resonance_point + detunings

annihilation_operators = []
for i in range(num_qubits):
    op_list = []
    for j in range(num_qubits):
        if i == j:
            op_list.append(qt.destroy(num_levels))
        else:
            op_list.append(qt.qeye(num_levels))
    a_i = qt.tensor(op_list)
    annihilation_operators.append(a_i)

H_detuned_0 = generate_Hamiltonian(annihilation_operators, num_levels, num_qubits, J, J, 0, frequencies)
H_resonant_0 = generate_Hamiltonian(annihilation_operators, num_levels, num_qubits, J, J, 0, [resonance_point]*num_qubits)

H_detuned_pi = generate_Hamiltonian(annihilation_operators, num_levels, num_qubits, J, -J, 0, frequencies)
H_resonant_pi = generate_Hamiltonian(annihilation_operators, num_levels, num_qubits, J, -J, 0, [resonance_point]*num_qubits)


label_to_Hamiltonian = {
    '0_flux_detuned': H_detuned_0,
    '0_flux_resonant': H_resonant_0,
    'pi_flux_detuned': H_detuned_pi,
    'pi_flux_resonant': H_resonant_pi
}

In [7]:
label_to_eigenvalues = {}
label_to_eigenstates = {}

In [8]:
for label in label_to_Hamiltonian:
    H = label_to_Hamiltonian[label]
    eigenvalues, eigenstates = H.eigenstates()
    label_to_eigenvalues[label] = eigenvalues
    label_to_eigenstates[label] = eigenstates

: 